# 13 · Açıklanabilirlik (XAI) ve zorlu görevler

Önce GPU açın (T4). Notebook 11 çalışmış olmalı. Süre yaklaşık 25-40 dk.

- **XAI:** Grad-CAM + Integrated Gradients (iki model), DeiT için dikkat yayılımı; 20 doğru + 12 yanlış örnek; rastgeleleştirme testi; silme eğrisi
- **Zorlu görev 1:** bozulma dayanıklılığı (4 bozulma × 5 şiddet; özgün, DeiT, kontrol)
- **Zorlu görev 2:** seçici tahmin (risk-kapsam; eşikler yalnızca doğrulamadan)

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
# Kod ve çıktılar Drive'da tek klasörde durur; ham veri oturum diskinde (/content/data) tutulur.
PROJECT = next((p for p in ('/content/drive/MyDrive/YZProje', '/content/drive/MyDrive/YZProje/YZProje')
                if os.path.isfile(p + '/src/config.py')), None)
assert PROJECT, "Drive'da YZProje/src/config.py bulunamadı: proje klasörünü MyDrive/YZProje olarak yerleştirin."
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = PROJECT + '/outputs'
os.environ['KNEE_DATA_ROOT'] = '/content/data'
print('Proje klasörü:', PROJECT)

In [ ]:
!pip install -q imagehash==4.3.2 tabulate==0.9.0 timm==1.0.30 scikit-image
import torch
print(torch.__version__, torch.cuda.is_available())

In [ ]:
# Görüntüler oturum diskinde yoksa: önce Drive'daki arşiv kopyası, o da yoksa bağlantıdan indirip arşivi Drive'a kaydeder.
from src import config as C, data_prep
print('Veri hazır:', data_prep.ensure_data())

## 1 · XAI

In [ ]:
from src import xai
print(xai.run())

In [ ]:
import pandas as pd
pd.set_option('display.width', 220)
print(pd.read_csv(xai.XAI / 'deletion_auc.csv').round(4))
print(pd.read_csv(xai.XAI / 'randomization_test.csv').round(3))

## 2 · Bozulma dayanıklılığı

In [ ]:
from src import challenges as H
H.robustness().round(4)

## 3 · Seçici tahmin

In [ ]:
df = H.selective(); H.captions()
df[df.güven == 'maks. olasılık'][['model','olasılık','hedef_kapsam(val)','test_kapsam','macro_f1','qwk','mae_grade','AURC']].round(4)

In [ ]:
from IPython.display import Image, display
from src import config as C
for f in ['fig10_silme_egrisi','fig11_rastgelelestirme','fig12_bozulma_dayaniklilik','fig13_secici_tahmin','fig08_xai_dogru_a','fig09_xai_hatali_a']:
    display(Image(str(C.FIG_DIR / f'{f}.png'), width=800))

## 4 · Çıktıları kaydet

In [ ]:
import shutil
from src import config as C
from src import xai, challenges as H
for d in (xai.XAI, H.CH): print(d, len(list(d.glob('*'))))
shutil.make_archive('/content/xai_ve_gorevler', 'zip', str(xai.OUT))
shutil.copy('/content/xai_ve_gorevler.zip', str(C.OUTPUT_DIR / 'xai_ve_gorevler.zip'))
shutil.make_archive('/content/sekiller_xai', 'zip', str(C.FIG_DIR))
shutil.copy('/content/sekiller_xai.zip', str(C.OUTPUT_DIR / 'sekiller_xai.zip'))
print('hazır')